# BA Audio Analysis – FF2: JSON vs. Natürlichsprachlicher Text

**Forschungsfrage 2:** Welches Übergabeformat – strukturiertes JSON oder natürlichsprachliche Beschreibung – führt zu empathischeren LLM-Antworten?

Experimentaufbau:
- 3 Szenarien: gestresst, traurig, neutral
- Je 3 Varianten: A (Baseline), B (JSON), C (natürlicher Text)
- 9 GPT-Aufrufe, identischer System-Prompt, Temperatur 0.3
- Automatische Bewertung durch ein zweites GPT (1–5 auf 3 Kriterien)
- Ergebnisse als pandas-DataFrame, JSON und CSV

## 1. Forschungsbezug

**FF2** untersucht, ob das *Format* der Zusatzinformationen (nicht ihr Inhalt) einen Unterschied macht:

- **Variante A** – Baseline: Nur der gesprochene Satz, keine Prosodieinfos
- **Variante B** – JSON: Satz + strukturiertes JSON (Stil v3: transcript + prosody + emotion)
- **Variante C** – Text: Satz + Prosodieinfos als natürlichsprachliche Beschreibung in Klammern

B und C enthalten **dieselbe inhaltliche Information** – nur das Format unterscheidet sich. So ist der Vergleich fair.

## 2. Setup und Konfiguration

Bibliotheken installieren (einmalig):
```bash
pip install openai python-dotenv pandas
```

In [1]:
import json
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY fehlt. Bitte in der .env-Datei setzen.")

client = OpenAI(api_key=api_key)

TEMPERATURE = 0.3
OUTPUT_DIR  = Path("../audio")

print("Setup erfolgreich")

Setup erfolgreich


## 3. Szenarien

Drei Szenarien mit vordefinierten, realistischen Prosodiewerten.
Die Werte orientieren sich an typischen Merkmalen der jeweiligen Emotion
(basierend auf den gemessenen Werten aus v2 für "gestresst").

Jedes Szenario enthält:
- `transcript`: Den gesprochenen Satz
- `prosody`: Prosodische Merkmale im v3-Format
- `emotion`: Label und Konfidenzwert
- `text_description`: Dieselbe Information als natürlicher Satz (für Variante C)

In [2]:
SCENARIOS = {
    "gestresst": {
        "transcript": "Ich bin so gestresst, ich schaffe das alles nicht mehr.",
        "prosody": {
            "duration_seconds":     4.2,
            "word_count":           10,
            "syllables_per_second": 4.8,
            "speech_rate_wpm":      143,
            "speech_rate_category": "normal",
            "silence_ratio_pct":    15.0,
            "volume_rms_mean":      0.082,
            "volume_category":      "laut"
        },
        "emotion": {"label": "angry", "score": 0.61},
        "text_description": (
            "Die Person klingt gestresst und angespannt, "
            "spricht laut und mit kaum Pausen."
        )
    },
    "traurig": {
        "transcript": "Ich weiß nicht, ich bin einfach so müde und traurig.",
        "prosody": {
            "duration_seconds":     5.8,
            "word_count":           10,
            "syllables_per_second": 2.9,
            "speech_rate_wpm":      103,
            "speech_rate_category": "slow",
            "silence_ratio_pct":    55.0,
            "volume_rms_mean":      0.018,
            "volume_category":      "leise"
        },
        "emotion": {"label": "sad", "score": 0.74},
        "text_description": (
            "Die Person klingt erschöpft und traurig, "
            "spricht langsam und sehr leise, macht viele Pausen."
        )
    },
    "neutral": {
        "transcript": "Ich habe heute meine Aufgaben erledigt und war auf der Arbeit.",
        "prosody": {
            "duration_seconds":     4.5,
            "word_count":           11,
            "syllables_per_second": 3.6,
            "speech_rate_wpm":      147,
            "speech_rate_category": "normal",
            "silence_ratio_pct":    30.0,
            "volume_rms_mean":      0.042,
            "volume_category":      "normal"
        },
        "emotion": {"label": "neutral", "score": 0.81},
        "text_description": (
            "Die Person klingt neutral und sachlich, "
            "spricht in normalem Tempo und normaler Lautstärke."
        )
    }
}

print(f"{len(SCENARIOS)} Szenarien definiert: {list(SCENARIOS.keys())}")

3 Szenarien definiert: ['gestresst', 'traurig', 'neutral']


## 4. Prompt-Konstruktion

Der System-Prompt ist für alle 9 Varianten identisch.
Der User-Prompt unterscheidet sich je nach Variante:

- **A**: Nur der Transkript-Text
- **B**: JSON-String mit transcript + prosody + emotion
- **C**: `"<Satz>" (<Textbeschreibung>)`

In [3]:
SYSTEM_PROMPT = (
    "Du bist ein einfühlsamer Assistent. "
    "Antworte auf die Aussage der Person kurz und empathisch auf Deutsch. "
    "Wenn du prosodische oder emotionale Informationen erhältst, berücksichtige diese in deiner Antwort."
)


def build_variant_a(scenario: dict) -> str:
    """Nur der gesprochene Satz – kein Kontext."""
    return scenario["transcript"]


def build_variant_b(scenario: dict) -> str:
    """Transkript + Prosodie + Emotion als JSON (v3-Format)."""
    payload = {
        "version":    "v3",
        "transcript": scenario["transcript"],
        "prosody":    scenario["prosody"],
        "emotion":    scenario["emotion"]
    }
    return json.dumps(payload, ensure_ascii=False)


def build_variant_c(scenario: dict) -> str:
    """Transkript + Prosodie/Emotion als natürlichsprachlicher Klammer-Text."""
    return f'"{scenario["transcript"]}" ({scenario["text_description"]})'


# Vorschau der drei Varianten am Beispiel "gestresst"
example = SCENARIOS["gestresst"]
print("=== Variante A ===")
print(build_variant_a(example))
print("\n=== Variante B ===")
print(build_variant_b(example))
print("\n=== Variante C ===")
print(build_variant_c(example))

=== Variante A ===
Ich bin so gestresst, ich schaffe das alles nicht mehr.

=== Variante B ===
{"version": "v3", "transcript": "Ich bin so gestresst, ich schaffe das alles nicht mehr.", "prosody": {"duration_seconds": 4.2, "word_count": 10, "syllables_per_second": 4.8, "speech_rate_wpm": 143, "speech_rate_category": "normal", "silence_ratio_pct": 15.0, "volume_rms_mean": 0.082, "volume_category": "laut"}, "emotion": {"label": "angry", "score": 0.61}}

=== Variante C ===
"Ich bin so gestresst, ich schaffe das alles nicht mehr." (Die Person klingt gestresst und angespannt, spricht laut und mit kaum Pausen.)


## 5. GPT-Experiment – 9 Aufrufe

Für jedes der 3 Szenarien werden die 3 Varianten abgerufen.
Modell: `gpt-4o-mini`, Temperatur: 0.3 (konsistent, reproduzierbar).

In [4]:
def ask_gpt(user_content: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content}
        ]
    )
    return response.choices[0].message.content.strip()


results = []

for scenario_name, scenario_data in SCENARIOS.items():
    variants = [
        ("A", build_variant_a(scenario_data)),
        ("B", build_variant_b(scenario_data)),
        ("C", build_variant_c(scenario_data))
    ]
    for variant_label, user_content in variants:
        answer = ask_gpt(user_content)
        results.append({
            "szenario":    scenario_name,
            "variante":    variant_label,
            "user_input":  user_content,
            "gpt_antwort": answer
        })
        print(f"[{scenario_name} / Variante {variant_label}]")
        print(answer)
        print("-" * 60)

[gestresst / Variante A]
Es tut mir leid zu hören, dass du so gestresst bist. Es ist verständlich, dass du dich überfordert fühlst. Vielleicht hilft es, einen Moment durchzuatmen und dir eine kleine Auszeit zu gönnen. Du bist nicht allein in dieser Situation.
------------------------------------------------------------
[gestresst / Variante B]
Es tut mir leid zu hören, dass du so gestresst bist. Es ist wirklich belastend, wenn man das Gefühl hat, alles nicht mehr zu schaffen. Wenn du magst, können wir darüber sprechen, was dich besonders belastet.
------------------------------------------------------------
[gestresst / Variante C]
Es tut mir leid zu hören, dass du so gestresst bist. Es klingt wirklich überwältigend. Hast du vielleicht einen Moment, um durchzuatmen oder dir eine kleine Auszeit zu gönnen?
------------------------------------------------------------
[traurig / Variante A]
Es tut mir leid zu hören, dass du dich so fühlst. Es ist völlig in Ordnung, müde und traurig zu sein

## 6. Automatische Bewertung

Ein zweites GPT bewertet jede der 9 Antworten auf drei Kriterien (Skala 1–5):

| Kriterium | Beschreibung |
|---|---|
| **emotionale_anerkennung** | Wird die Emotion der Person explizit anerkannt? |
| **situative_angemessenheit** | Passt die Antwort zur beschriebenen Situation? |
| **sprachliche_waerme** | Wie warm und fürsorglich klingt die Formulierung? |

Temperatur 0.0 für maximale Konsistenz der Bewertung.

In [5]:
EVAL_SYSTEM_PROMPT = """Du bewertest KI-generierte empathische Antworten.
Gegeben sind die Originaleingabe an die KI und ihre Antwort.
Bewerte die Antwort auf einer Skala von 1 bis 5 nach diesen drei Kriterien:

- emotionale_anerkennung: Erkennt die Antwort die Emotion der Person an? (1=gar nicht, 5=sehr gut)
- situative_angemessenheit: Ist die Antwort für die Situation passend? (1=unpassend, 5=sehr passend)
- sprachliche_waerme: Wie warm und fürsorglich klingt die Sprache? (1=kalt/neutral, 5=sehr warm)

Antworte ausschliesslich mit diesem JSON, ohne Erklärungen:
{"emotionale_anerkennung": X, "situative_angemessenheit": X, "sprachliche_waerme": X}"""


def evaluate_answer(user_input: str, gpt_answer: str) -> dict:
    eval_prompt = f"Originaleingabe:\n{user_input}\n\nAntwort der KI:\n{gpt_answer}"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "system", "content": EVAL_SYSTEM_PROMPT},
            {"role": "user",   "content": eval_prompt}
        ]
    )
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)


for entry in results:
    scores = evaluate_answer(entry["user_input"], entry["gpt_antwort"])
    entry["emotionale_anerkennung"]   = scores["emotionale_anerkennung"]
    entry["situative_angemessenheit"] = scores["situative_angemessenheit"]
    entry["sprachliche_waerme"]       = scores["sprachliche_waerme"]
    entry["gesamt"] = sum(scores.values())
    print(f"[{entry['szenario']} / {entry['variante']}] → {scores}")

[gestresst / A] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 4}
[gestresst / B] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 4}
[gestresst / C] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 4}
[traurig / A] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 5}
[traurig / B] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 5}
[traurig / C] → {'emotionale_anerkennung': 5, 'situative_angemessenheit': 5, 'sprachliche_waerme': 5}
[neutral / A] → {'emotionale_anerkennung': 4, 'situative_angemessenheit': 5, 'sprachliche_waerme': 4}
[neutral / B] → {'emotionale_anerkennung': 4, 'situative_angemessenheit': 5, 'sprachliche_waerme': 4}
[neutral / C] → {'emotionale_anerkennung': 3, 'situative_angemessenheit': 4, 'sprachliche_waerme': 4}


## 7. Ergebnistabelle

Alle 9 Ergebnisse zusammengefasst in einer pandas-Tabelle.
Die Spalte **Gesamt** ist die Summe der drei Kriterien (max. 15 Punkte).

In [6]:
df = pd.DataFrame(results)[[
    "szenario", "variante", "gpt_antwort",
    "emotionale_anerkennung", "situative_angemessenheit",
    "sprachliche_waerme", "gesamt"
]]

df.columns = [
    "Szenario", "Variante", "GPT-Antwort",
    "Em. Anerkennung", "Sit. Angemessenheit",
    "Spr. Wärme", "Gesamt"
]

pd.set_option("display.max_colwidth", 80)
df

,Szenario,Variante,GPT-Antwort,Em. Anerkennung,Sit. Angemessenheit,Spr. Wärme,Gesamt
0,gestresst,A,"Es tut mir leid zu hören, dass du so gestresst bist. Es ist verständlich, da...",5,5,4,14
1,gestresst,B,"Es tut mir leid zu hören, dass du so gestresst bist. Es ist wirklich belaste...",5,5,4,14
2,gestresst,C,"Es tut mir leid zu hören, dass du so gestresst bist. Es klingt wirklich über...",5,5,4,14
3,traurig,A,"Es tut mir leid zu hören, dass du dich so fühlst. Es ist völlig in Ordnung, ...",5,5,5,15
4,traurig,B,"Es tut mir leid zu hören, dass du dich so müde und traurig fühlst. Es ist vo...",5,5,5,15
5,traurig,C,"Es tut mir leid zu hören, dass du dich so fühlst. Es ist verständlich, dass ...",5,5,5,15
6,neutral,A,"Das klingt nach einem produktiven Tag! Es ist immer schön, wenn man seine Au...",4,5,4,13
7,neutral,B,"Es klingt, als hättest du einen produktiven Tag gehabt. Gut gemacht!",4,5,4,13
8,neutral,C,"Das klingt nach einem produktiven Tag! Gut, dass du deine Aufgaben erledigen...",3,4,4,11


## 8. Aggregierte Auswertung

Durchschnittswerte pro Variante über alle 3 Szenarien.

In [7]:
summary = (
    df.groupby("Variante")[["Em. Anerkennung", "Sit. Angemessenheit", "Spr. Wärme", "Gesamt"]]
    .mean()
    .round(2)
    .sort_values("Gesamt", ascending=False)
)

print("Durchschnitt pro Variante (über alle 3 Szenarien):")
print(summary)

Durchschnitt pro Variante (über alle 3 Szenarien):
          Em. Anerkennung  Sit. Angemessenheit  Spr. Wärme  Gesamt
Variante                                                          
A                    4.67                 5.00        4.33   14.00
B                    4.67                 5.00        4.33   14.00
C                    4.33                 4.67        4.33   13.33


## 9. Ergebnisse speichern

- `ff2_results.csv` – tabellarische Übersicht für Auswertung in Excel / R
- `ff2_results.json` – vollständige Rohdaten inkl. Prompts für Reproduzierbarkeit

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path  = OUTPUT_DIR / "ff2_results.csv"
json_path = OUTPUT_DIR / "ff2_results.json"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment":   "FF2 – JSON vs. natürlichsprachlicher Text",
            "temperature":  TEMPERATURE,
            "system_prompt": SYSTEM_PROMPT,
            "results":      results
        },
        f, indent=2, ensure_ascii=False
    )

print(f"Gespeichert: {csv_path}")
print(f"Gespeichert: {json_path}")

Gespeichert: ..\audio\ff2_results.csv
Gespeichert: ..\audio\ff2_results.json


## 10. Nächste Schritte

- [ ] Experiment mit echten Audiodateien wiederholen (statt vordefinierten Werten)
- [ ] Mehrere Durchläufe mit verschiedenen Temperaturen (0.0, 0.3, 0.7, 1.0)
- [ ] Menschliche Bewertung zur Validierung der GPT-Evaluierung hinzuziehen
- [ ] Ergebnisse statistisch auswerten (Mittelwertvergleich A vs. B vs. C)
- [ ] Erkenntnisse in BA-Kapitel "Experimentdesign" überführen